In [ ]:
import xarray as xr
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm.auto import tqdm
import uqct

In [ ]:
import glob

def read_dataset(path, dataset=None):
    # recursively read all netcdf files in path
    nc_files = sorted(glob.glob(f"{path}/**/*.nc", recursive=True))

    datasets = {}
    attrs = []
    for f in nc_files:
        print(f)
        ds = xr.open_dataset(f)
        print(ds.attrs)

        if dataset is not None:
            if ds.attrs["dataset"] != dataset:
                continue
        attrs.append(ds.attrs)
        experiment_id = ds.attrs['experiment_id']
        ds['beta'] = ds['seq_nll'].cumsum(dim='step')
        datasets[experiment_id] = ds
    df = pd.DataFrame(attrs)
    df = df.set_index('experiment_id')
    return datasets, df


# datasets, attrs = read_dataset("../results/2026-01-13/")
# datasets, attrs = read_dataset("../results/2026-01-13_30/")
# datasets, attrs = read_dataset("../results/2026-01-16/")
# datasets, attrs = read_dataset("../results/2026-01-16_30/")
# datasets, attrs = read_dataset("../results/2026-01-19_fix/")
datasets, attrs = read_dataset("../results/2026_01_19_fix/")
# datasets, attrs = read_dataset("../results/inverse_crime_test/")
# datasets, attrs = read_dataset("../results/rep1000/")
attrs
_attrs = attrs.copy()
attrs

In [ ]:
# # attrs = _attrs.loc[_attrs['initial_intensity'] == str(1e6)]
# attrs = _attrs.loc[(_attrs['model'] != 'mle') | ((_attrs['iterative_num_gradient_steps'] == '100') & (_attrs['iterative_lr'] == str(1e-1)) )]

# attrs = _attrs.loc[(_attrs['model'] != 'cond_diffusion') | (_attrs['guidance_num_gradient_steps'] == '20') ]
attrs = _attrs.loc[(_attrs['model'] != 'diffusion') | (_attrs['guidance_num_gradient_steps'] == '20') ]
# attrs = _attrs.loc[(_attrs['model'] != 'diffusion') | (_attrs['diffusion_num_inference_steps'] == '100') ]
# attrs = _attrs.loc[(_attrs['model'] != 'cond_diffusion') | (_attrs['num_samples'] == '10') ]
# attrs = _attrs.loc[(_attrs['model'] != 'cond_diffusion') | (_attrs['bypass_inverse_crime'] == 'True') ]
# attrs = _attrs.loc[(_attrs['batch_size'] == '10') ]

attrs = attrs.loc[(_attrs['initial_intensity'] == str(1e6)) ]


print("Before filtering:", len(_attrs), "After filtering:", len(attrs))
attrs.sort_values(['model', 'dataset'])

In [ ]:
line_kwargs = {
    'fbp' :{'color': 'C0', 'linestyle': '-', 'label': 'FBP'},
    'unet':{'color': 'C1', 'linestyle': '-', 'label': 'U-Net'},
    'mle' :{'color': 'C3', 'linestyle': '-', 'label': 'MLE'},
    'map' :{'color': 'C5', 'linestyle': '-', 'label': 'MAP'},
    'unet_ensemble':{'color': 'C1', 'linestyle': '--', 'label': 'U-Net Ensemble'},
    'cond_diffusion':{'color': 'C2', 'linestyle': '-', 'label': 'Cond. Diffusion'},
    'cond_diffusion_mix':{'color': 'C2', 'linestyle': '--', 'label': 'Cond. Diffusion Mix'},
    'diffusion':{'color': 'C4', 'linestyle': '-', 'label': 'Diffusion'},
    'diffusion_mix':{'color': 'C4', 'linestyle': '--', 'label': 'Diffusion Mix'},
    'gt': {'color': 'black', 'linestyle': ':', 'label': 'Ground Truth'},
    'gt_rot1.0': {'color': 'black', 'linestyle': ':', 'label': 'Ground Truth'},
    'gt_rot0.5': {'color': 'black', 'linestyle': ':', 'label': 'Ground Truth'},
}

def find_experiment(dataset, model):
    model = model_sel = model
    if model == 'cond_diffusion_mix':
        model, model_sel = 'cond_diffusion', 'cond_diffusion_mix'
    if model == 'diffusion_mix':
        model, model_sel = 'diffusion', 'diffusion_mix'
    if model.startswith('gt'):
        if 'rot' in model:
            rotation = model.split('rot')[1]
        else:
            rotation = None
        model = "gt"
    if model == 'gt':
        if rotation is None:
            if 'rotation' in attrs.columns:
                matching_experiments = attrs.loc[(attrs['model'] == model) & (attrs['dataset'] == dataset) & (attrs['rotation'].isna() | (attrs['rotation'] == 'None'))]
            else:
                matching_experiments = attrs.loc[(attrs['model'] == model) & (attrs['dataset'] == dataset)]
        else:
            matching_experiments = attrs.loc[(attrs['model'] == model) & (attrs['dataset'] == dataset) & (attrs['rotation'] == rotation)]
    else:
        matching_experiments = attrs.loc[(attrs['model'] == model) & (attrs['dataset'] == dataset)]
    if len(matching_experiments) == 0:
        raise ValueError(f"No experiment found for model {model} and dataset {dataset}.")
    elif len(matching_experiments) > 1:
        print(f"WARNING Multiple experiments found for model {model} and dataset {dataset}. Using the first one.")

    experiment_id = matching_experiments.index[0]
    ds = datasets[experiment_id].sel(model=model_sel)
    return ds, experiment_id

def plot_psnr(ax, dataset, model, metric='psnr', plot_range=True):
    try:
        ds, experiment_id = find_experiment(dataset, model)
    except ValueError as e:
        # print(e)
        return

    if metric in ['psnr', 'nll', 'beta', 'ssim']:
        y = ds[metric]
    elif metric == 'beta_diff':
        ds_gt, _ = find_experiment(dataset, 'gt')
        y = ds["beta"] - ds_gt["beta"]
    elif metric == 'beta_diff_fbp':
        ds_fbp, _ = find_experiment(dataset, 'fbp')
        y = ds["beta"] - ds_fbp["beta"]
    elif metric == 'beta_diff_1.0':
        ds_gt, _ = find_experiment(dataset, 'gt_rot1.0')
        y = ds["beta"] - ds_gt["beta"]
    elif metric == 'beta_diff_0.5':
        ds_gt, _ = find_experiment(dataset, 'gt_rot0.5')
        y = ds["beta"] - ds_gt["beta"]
    else:
        raise ValueError(f"Unknown metric {metric}")
    y_mean = y.mean(dim=['index', 'seed']).squeeze().values
    y_min = y.min(dim=['index', 'seed']).squeeze().values
    y_max = y.max(dim=['index', 'seed']).squeeze().values
    intensity = ds.sel(seed=0, index=0)['intensity']
    x = intensity.cumsum()
    plot_kwargs_without_label = {k: v for k, v in line_kwargs[model].items() if k != 'label'}
    ax.plot(x, y_mean, **line_kwargs[model])
    ax.scatter(x, y_mean, marker='x', **plot_kwargs_without_label)
    if plot_range:
        # Remove 'label' from kwargs for fill_between to avoid duplicate legend entries
        ax.fill_between(x, y_min, y_max, alpha=0.2, **plot_kwargs_without_label)
    ax.set_xscale("log")
    ax.set_title(f"{dataset} dataset")


models = ['fbp', 'mle', 'map', 'unet', 'cond_diffusion', 'cond_diffusion_mix', 'diffusion', 'diffusion_mix', 'unet_ensemble']
# psnr
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
    for model in models:
        plot_psnr(ax, dataset, model, metric='psnr')
    ax.set_xlabel("Total Intensity")
    ax.set_ylabel("PSNR (dB)")
    ax.legend()

# ssim
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
    for model in models:
        plot_psnr(ax, dataset, model, metric='ssim')
    ax.set_xlabel("Total Intensity")
    ax.set_ylabel("SSIM")
    ax.legend()


# nll
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
    for model in models:
        plot_psnr(ax, dataset, model, metric='nll')
    ax.set_xlabel("Total Intensity")
    ax.set_ylabel("NLL")
    ax.legend()

# beta
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
    for model in models + ['gt']:
        plot_psnr(ax, dataset, model, metric='beta')
    ax.set_xlabel("Total Intensity")
    ax.set_ylabel("beta")
    ax.set_yscale('log')
    ax.legend()

# beta diff
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
    for model in models:
        plot_psnr(ax, dataset, model, metric='beta_diff')
    ax.set_xlabel("Total Intensity")
    ax.set_ylabel("beta - beta(gt)")
    ax.set_ylim([0, 100000])
    ax.legend()

# beta zoomed
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
    for model in models:
        plot_psnr(ax, dataset, model, metric='beta_diff')
    ax.set_xlabel("Total Intensity")
    ax.set_ylabel("Zoomed: beta - beta(gt)")
    ax.set_ylim([-1000, 5000])
    ax.axhline(0, color='k')
    ax.legend()

# # beta diff fbp
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
#     for model in models:
#         plot_psnr(ax, dataset, model, metric='beta_diff_fbp')
#     ax.set_xlabel("Total Intensity")
#     ax.set_ylabel("beta - beta(fbp)")
#     ax.legend()

# # beta diff with rotation 1.0
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# for ax, dataset in zip(axes, ['lamino', 'composite', 'lung']):
#     for model in models:
#         plot_psnr(ax, dataset, model, metric='beta_diff_1.0', plot_range=False)
#     ax.set_xlabel("Total Intensity")
#     ax.set_ylabel("beta - beta(gt_rot1.0)")
#     ax.legend()

In [ ]:
find_experiment('lamino', 'unet')[0].sel(index=0)['beta']

In [ ]:
find_experiment('lamino', 'gt')[0].sel(index=0).attrs

In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from typing import Any
import argparse
import os

from uqct.datasets.utils import get_dataset
from uqct.metrics import get_metrics
from uqct.ct import fbp, nll, Experiment, Tomogram, anscombe_transform, radon, sinogram_from_counts, poisson, sample_observations, circular_mask

import torch.nn.functional as F
import astra

from uqct.models.diffusion import load_unet as load_diffusion_unet
from diffusers.models.unets.unet_2d import UNet2DModel
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from uqct.models.guided_diffusion import GradientGuidance, GuidedDiffusionPipeline
import xarray as xr

from uqct.training.unet import N_BINS_HR
from typing import NamedTuple


from uqct.evaluation.eval_dense import schedule_exponential, schedule_uniform, ObservationDataset, fbp_recon

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Args(NamedTuple):
    model: str = "fbp"  # "unet", "diffusion", "cond_diffusion"
    dataset: str = "lung"
    num_images: int = 10
    batch_size: int = 2
    seeds: list[int] = [0]
    schedule: str = "exponential"  # "uniform" or "exponential"
    total_intensity: float = 1e9
    num_steps: int = 10
    init_fraction: float = None
    initial_intensity: float = 1e7
    base: float = 1.1
    num_samples: int = 5
    diffusion_num_inference_steps: int = 100
    guidance_end: int = 10
    guidance_num_gradient_steps: int = 10
    guidance_lr: float = 1e-3

dataset = "lamino"
model = "cond_diffusion"
idx = 0
seed = 0

ds, _ = find_experiment(dataset, model)

args = Args(
    model=model,
    seeds=[seed], 
    dataset=dataset, 
    initial_intensity=float(ds.attrs['initial_intensity']), 
    total_intensity=float(ds.attrs['total_intensity']), 
    num_steps=int(ds.attrs['num_steps']),
    diffusion_num_inference_steps=int(ds.attrs['diffusion_num_inference_steps']),
    guidance_end=int(ds.attrs['guidance_end']),
    guidance_num_gradient_steps=int(ds.attrs['guidance_num_gradient_steps']),
    guidance_lr=float(ds.attrs['guidance_lr']),
    )
print(args)
print(f"{args.total_intensity:e}")
print(f"{args.initial_intensity:e}")

# get dataset
_ , test_set = get_dataset(args.dataset, True)
if args.num_images is not None:
    test_set = torch.utils.data.Subset(test_set, list(range(args.num_images)))

# set up observation parameters
num_angles = 200
angles = torch.from_numpy(np.linspace(0, 180, num_angles, endpoint=False)).float().to(device)
if args.schedule == "uniform":
    schedule = schedule_uniform(
        args.total_intensity, args.num_steps, init_fraction=args.init_fraction, device=device
    )
elif args.schedule == "exponential":
    schedule = schedule_exponential(
        args.total_intensity, args.num_steps, initial_intensity=args.initial_intensity, device=device
    )
total_intensities = schedule.clone()
print(total_intensities.cumsum(dim=0))
schedule = schedule.reshape(-1, 1, 1, 1).expand(-1, 1, len(angles), 1) / len(angles) / 256
# schedule_steps = np.arange(schedule.shape[1])

# set up observation dataset and dataloader
obs_dataset = ObservationDataset(test_set, seeds=args.seeds, intensities=schedule, angles=angles)
# obs_dataloader = torch.utils.data.DataLoader(
#     obs_dataset, batch_size=args.batch_size, shuffle=False, num_workers=0
# )

_, _, image, data = obs_dataset.get(idx, seed)


In [ ]:
image_lr = F.interpolate(image.unsqueeze(0), size=(128, 128), mode='area').squeeze(0)
image_lr

In [ ]:
mask = circular_mask(image_lr.shape[-1], device=image_lr.device)
image_lr_masked = image_lr * mask

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(12, 3))
ax1.imshow(image.squeeze().cpu(), cmap='gray')
ax2.imshow(image_lr.squeeze().cpu(), cmap='gray')
ax3.imshow(image_lr_masked.squeeze().cpu(), cmap='gray')
ax4.imshow( (image_lr - image_lr_masked).squeeze().cpu(), cmap='viridis')

In [ ]:
print(total_intensities.cumsum(dim=0))

In [ ]:
from diffusers.models.unets.unet_2d import UNet2DModel
from uqct.evaluation.eval_dense import UNetRecon, load_unet, CondDiffusionRecon
from uqct.models.diffusion import load_unet as load_diffusion_unet


print(model)

if model == "fbp":
    recon = fbp_recon
elif model == "unet":
    ckpt_path = Path(f'/mydata/chip/shared/checkpoints/uqct/unet_dense/unet_dense_128_{args.dataset}_0.pt')
    print(f"Loading UNet checkpoint from {ckpt_path}")
    unet = load_unet(ckpt_path, sparse=False).to(device).eval()
    recon = UNetRecon(unet)
elif model == "cond_diffusion":
    ckpt_path = Path(f"/mydata/chip/shared/checkpoints/uqct/diffusion/ddpm_conditional_128_{args.dataset}.pt")
    unet_diffusion = load_diffusion_unet(ckpt_path, cond=True).to(device).eval()
    scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")
    print(args.num_samples)
    recon = CondDiffusionRecon(
        unet_diffusion,
        scheduler,
        num_samples=args.num_samples,
        num_inference_steps=args.diffusion_num_inference_steps,
        guidance_start=1000,
        guidance_end=args.guidance_end,
        guidance_num_gradient_steps=args.guidance_num_gradient_steps,
        guidance_lr=args.guidance_lr
    )

In [ ]:

steps = [1, 2, 3, 4, 8, 10]

recon = CondDiffusionRecon(
    unet_diffusion,
    scheduler,
    num_samples=args.num_samples,
    num_inference_steps=args.diffusion_num_inference_steps,
    guidance_start=1000,
    guidance_end=args.guidance_end,
    guidance_num_gradient_steps=args.guidance_num_gradient_steps,
    # guidance_num_gradient_steps=1,
    guidance_lr=args.guidance_lr
    # guidance_lr=1e-8
)

with torch.no_grad():
    # add batch dimension and sum total intensity and observations
    data_cumsum = data.unsqueeze(0).cumsum(dim=1)  
    schedule_cumsum = schedule.unsqueeze(0).cumsum(dim=1)
    # total_intensities = schedule_cumsum.expand(data_cumsum.shape).sum(dim=(-1, -2))
    # print(data_cumsum.shape, schedule_cumsum.shape)
    with torch.no_grad():
        pred = [recon(data_cumsum[:, i], schedule_cumsum[:, i], angles) for i in tqdm(steps)]
        print(pred[0].shape)
        if args.model in ["diffusion", "cond_diffusion"]:
            pred = torch.stack(pred, dim=2)
        else:
            pred = torch.stack(pred, dim=1)
pred  # shape 11, 1, 128, 128 for (num_steps, channels, height, width)
# pred_incr = pred[0, 1:]
# pred_incr

In [ ]:
fig, axes = plt.subplots(len(steps), len(pred), figsize=(2 * len(pred), 2 * len(steps)))


for j in range(len(steps)):
    for i in range(len(pred)):
        axes[j, i].imshow(pred[i, 0, j].squeeze().cpu(), cmap='gray')
        axes[j, i].set_title(f"Step {steps[i]}")
        axes[j, i].axis('off')

In [ ]:
img_pred = pred[0,0,-1]
cum_nll_pred = nll(img_pred, data, schedule, angles).sum(dim=[-1, -2, -3])[1:].cumsum(dim=0).cpu()
beta = ds.sel(seed=seed, index=idx)['beta'].squeeze().values

plt.plot(cum_nll_pred, label='predicted beta')
plt.plot(beta[1:], label='true beta')
plt.legend()

In [ ]:
def guidance_loss_beta(counts, intensities, angles, beta, data_steps, schedule_steps, l=5.):
    """
    Define a loss function for the diffusion model.
    This can be used to guide the diffusion process.
    """
    data_shape = counts.shape[:-2]
    circle_mask = circular_mask(counts.shape[-1], device=counts.device)
    def loss_fn(image):
        img_shape = image.shape[-2:]
        image = image.view(-1, *data_shape, *img_shape)
        image = ((image + 1.0)/2).clip(0, 1)
        image = image * circle_mask

        # note batch dims are a bit mixed up here: image does not have a step dimension, data_step has a batch dimension but not a sampling dimension
        # if we want both batch and step dims, we need to expand image
        # full_nll = nll(image, data_steps, schedule_steps, angles, l=l).sum(dim=[-1,-2, -3])

        step_nll =  nll(image, data_steps, schedule_steps, angles, l=l).sum(dim=[-1,-2, -3])
        first_step_nll = step_nll[:,0]
        remaining_step_nll = step_nll[:,1:]
        beta_loss = remaining_step_nll.sum(dim=-1)
        loss = torch.abs(beta_loss - beta)# + first_step_nll
        # print(beta_loss - beta)
        return loss.sum()  # remaining dimensions (samples, batch, steps)
    return loss_fn


recon_beta = CondDiffusionRecon(
    unet_diffusion,
    scheduler,
    num_samples=args.num_samples,
    num_inference_steps=args.diffusion_num_inference_steps,
    guidance_start=1000,
    guidance_end=0,
    guidance_num_gradient_steps=50,
    guidance_lr=1e-3,
    guidance_loss=guidance_loss_beta
)

steps = [1, 2, 3, 4, 5, 10, 20, 30]

with torch.no_grad():
    # add batch dimension and sum total intensity and observations
    data_cumsum = data.unsqueeze(0).cumsum(dim=1)  
    schedule_cumsum = schedule.unsqueeze(0).cumsum(dim=1)
    data_steps = data.unsqueeze(0)
    schedule_steps = schedule.unsqueeze(0)
    beta = beta = ds["beta"].sel(index=0, seed=0).squeeze().values
    beta = torch.tensor(beta, device=device, dtype=torch.float32)
    print(beta.shape)
    # total_intensities = schedule_cumsum.expand(data_cumsum.shape).sum(dim=(-1, -2))
    # print(data_cumsum.shape, schedule_cumsum.shape)
    with torch.no_grad():
        pred_beta = [recon_beta(data_cumsum[:, i], schedule_cumsum[:, i], angles, beta=beta[i], data_steps=data_steps[:, :i+1], schedule_steps=schedule_steps[:, :i+1]) for i in tqdm(steps)]
        print(pred_beta[0].shape)
        if args.model in ["diffusion", "cond_diffusion"]:
            pred_beta = torch.stack(pred_beta, dim=2)
        else:
            pred_beta = torch.stack(pred_beta, dim=1)

In [ ]:
pred_beta.unsqueeze(3), data, pred

In [ ]:
beta = ds["beta"].sel(index=0, seed=0).squeeze().values
beta = torch.tensor(beta, device=device, dtype=torch.float32)
x = total_intensities.cumsum(dim=0)[steps].squeeze().cpu()
x = [*range(len(steps))]


nll_cum_beta = nll(pred_beta.unsqueeze(3), data.unsqueeze(0), schedule, angles).sum(dim=[-1, -2, -3])[:, :, :, 1:].cumsum(dim=-1)
idx = torch.tensor(steps, dtype=torch.long, device=nll_cum_beta.device) - 1  # (8,)
idx = idx.view(1, 1, -1, 1).expand(nll_cum_beta.shape[0], nll_cum_beta.shape[1], nll_cum_beta.shape[2], 1)  # (5, 1, 8, 1)
nll_cum_beta = torch.gather(nll_cum_beta, dim=3, index=idx).squeeze(3)  # (5, 1, 8)
print(nll_cum_beta.shape)

nll_cum = nll(pred.unsqueeze(3), data.unsqueeze(0), schedule, angles).sum(dim=[-1, -2, -3])[:, :, :, 1:].cumsum(dim=-1)
idx = torch.tensor(steps, dtype=torch.long, device=nll_cum.device) - 1  # (8,)
idx = idx.view(1, 1, -1, 1).expand(nll_cum.shape[0], nll_cum.shape[1], nll_cum.shape[2], 1)  # (5, 1, 8, 1)
nll_cum = torch.gather(nll_cum, dim=3, index=idx).squeeze(3)  # (5, 1, 8)
# print(nll_cum)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(x, (nll_cum_beta).squeeze().T.cpu().numpy())
plt.gca().set_prop_cycle(None)
ax1.plot(x, (nll_cum).squeeze().T.cpu().numpy(), linestyle=':')
ax1.plot(x, (beta[steps]).squeeze().T.cpu().numpy(), linestyle='--', color='black')

# ax2.plot(((nll_cum_beta - beta[1:])/beta[1:]).squeeze().T.cpu().numpy())
ax2.plot(x, ((nll_cum_beta - beta[steps])).squeeze().T.cpu().numpy())
plt.gca().set_prop_cycle(None)
ax2.plot(x, ((nll_cum - beta[steps])).squeeze().T.cpu().numpy(), linestyle=':')

In [ ]:
total_intensities.squeeze().cumsum(dim=0)

In [ ]:
print("test")

In [ ]:

_num_samples = 3
fig, axes = plt.subplots(len(steps), 2 * _num_samples + 4, figsize=((2 * _num_samples + 4)* 3, 3 * len(steps)))
for i, (step_i, axs) in enumerate(zip(range(len(steps)), axes)):
    step = steps[step_i]
    _intensity = total_intensities[:step].sum().item()
    for j in range(_num_samples):
        axs[j].imshow(pred[j, 0, step_i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
        axs[j].set_title(f"Step {step} Sample {j}", fontsize=11)
        axs[j].axis('off')

        axs[j + _num_samples].imshow(pred_beta[j, 0, step_i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
        axs[j + _num_samples].set_title(f"Step {step} Sample {j} (beta)", fontsize=11)
        axs[j + _num_samples].axis('off')

    std_image = torch.std(pred[:, 0, step_i, 0], dim=0)
    std_image_beta = torch.std(pred_beta[:, 0, step_i, 0], dim=0)
    mean_std_image = torch.mean(std_image)
    mean_std_image_beta = torch.mean(std_image_beta)
    mean_image = torch.mean(pred[:, 0, step_i, 0], dim=0)
    abs_error = torch.abs(mean_image - image_lr[0])

    axs[-4].imshow(std_image_beta.cpu(), cmap='hot')
    axs[-4].set_title(f"std (beta)\n {mean_std_image_beta.item()}", fontsize=11)
    axs[-4].axis('off')

    axs[-3].imshow(std_image.cpu(), cmap='hot')
    axs[-3].set_title(f"std\n {mean_std_image.item()}", fontsize=11)
    axs[-3].axis('off')
    
    axs[-2].imshow(abs_error.cpu(), cmap='hot')
    axs[-2].set_title(f"Absolute Error\n", fontsize=11)
    axs[-2].axis('off')

    axs[-1].imshow(image_lr[0].cpu(), cmap='gray', vmin=0, vmax=1)
    axs[-1].set_title(f"Ground Truth\nTotal Intensity {_intensity:.1e}", fontsize=11)
    axs[-1].axis('off')

fig.subplots_adjust()
fig.tight_layout()

In [ ]:
beta = ds["beta"].sel(index=idx, seed=seed).squeeze().values
beta

In [ ]:
# def total_nll(image, sinogram, angles, intensity):
#     image = image.expand(schedule.shape[0], -1, -1, -1)
#     return nll(image[:, :-1].contiguous(), sinogram[:, 1:], intensity[1:], angles).sum((-1,-2)).squeeze(-1).sum()

# def dist(im1, im2):
#     return F.mse_loss(im1, im2)

# image = torch.nn.Parameter(pred[-1:].clone().detach().to(device))

# solve max dist(image, pred[-1]) s.t. total_nll(image, sinogram, angles, total_intensities) <= beta

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# --- pick target + gt (low-res) ---
pred_final = pred[-1].detach().to(device)              # (1,128,128) or (1,1,128,128) depending on your fbp output
if pred_final.dim() == 3:
    pred_final = pred_final.unsqueeze(0)               # -> (1,1,128,128)
assert pred_final.shape[-2:] == (128, 128)

# ground truth image from ObservationDataset (high-res), downsample to 128
gt_lr = F.interpolate(image.unsqueeze(0), size=(128, 128), mode="area").detach().to(device)  # (1,1,128,128)

# --- beta from xarray (final step) ---
beta = ds["beta"].sel(index=idx, seed=seed).squeeze().values
beta = torch.tensor(beta, device=device, dtype=torch.float32)

# --- incremental measurements: match eval_dense convention (pred[:-1] vs data[1:]) ---
counts_incr = data[1:].detach().to(device)     # (T-1,1,angles,det)
intens_incr = schedule[1:].detach().to(device)     # (T-1,1,angles,1)
beta_incr = beta[1:]
Tminus1 = counts_incr.shape[0]

def total_nll_const_image(img_1x1hw: torch.Tensor) -> torch.Tensor:
    """
    Total incremental NLL across steps (like eval_dense):
      sum_t nll(img, counts[t], intens[t]) over t=1..T-1, then sum over angles/detectors.
    """
    assert img_1x1hw.shape[:2] == (1, 1)
    img_rep = img_1x1hw.expand(Tminus1, -1, -1, -1).contiguous()  # (T-1,1,H,W)
    # nll returns per-ray tensor; sum over (angles, det) -> (T-1,1) -> scalar
    nll_per = nll(img_rep, counts_incr, intens_incr, angles).sum(dim=(-1, -2)).squeeze(-1)
    return nll_per

def delta_total_nll(img_1x1hw: torch.Tensor) -> torch.Tensor:
    return total_nll_const_image(img_1x1hw) - total_nll_gt

def dist_fn(img_1x1hw: torch.Tensor, ref_1x1hw: torch.Tensor) -> torch.Tensor:
    # maximize MSE distance (simple + stable)
    return F.mse_loss(img_1x1hw, ref_1x1hw)

with torch.no_grad():
    total_nll_gt = total_nll_const_image(gt_lr).detach()

step = 99
print("beta =", beta_incr[step])
print("total_nll(gt_lr) =", total_nll_gt)

In [ ]:
counts_incr.shape, intens_incr.shape, angles.shape

In [ ]:
# --- simple projection: minimize violation ReLU(ΔNLL - beta)^2 ---
@torch.no_grad()
def _clamp01_(x: torch.Tensor):
    x.clamp_(0.0, 1.0)

# --- circular mask constraint (forces image=0 outside reconstruction circle) ---
# uses uqct.ct.circular_mask that you imported earlier
mask_hw = circular_mask(128, device=device).float()          # (128,128)
mask_1x1hw = mask_hw[None, None, :, :]                      # (1,1,128,128)

@torch.no_grad()
def clamp_and_mask_(x: torch.Tensor, lo: float = 0.0, hi: float = 1.0) -> None:
    x.clamp_(lo, hi)
    x.mul_(mask_1x1hw)  # enforce x=0 outside the circle

# def project_to_confidence_simple(
#     img_param: torch.nn.Parameter,
#     steps: int = 200,
#     lr: float = 1e-2,
#     tol: float = 1e-4,
# ) -> None:
#     opt = torch.optim.Adam([img_param], lr=lr)
#     for _ in range(steps):
#         opt.zero_grad()
#         _total_nll = total_nll_const_image(img_param).cumsum(dim=0)
#         # d = delta_total_nll(img_param)
#         viol = torch.relu(_total_nll - beta_incr)
#         if float(viol.detach().cpu()) <= tol:
#             break
#         loss = viol * viol
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_([img_param], 1.0)
#         opt.step()
#         with torch.no_grad():
#             clamp_and_mask_(img_param)


def compute_maximizer(initial_x, ref_x, print_step=step):
    # --- outer loop: ascend distance, then project back into the set ---
    torch.manual_seed(0)

    x = torch.nn.Parameter(initial_x.clone().detach())  # init at pred (you can also try random)
    outer_opt = torch.optim.Adam([x], lr=1e-3)

    history = {"dist": [], "delta": [], "viol": [], "nll" : []}
    outer_steps = 1500 * 2

    for it in range(outer_steps):
        outer_opt.zero_grad()
        dist = (x - ref_x).pow(2).mean(dim=(-1, -2, -3)) 

        nll_x = nll(x.unsqueeze(1), counts_incr, intens_incr, angles).sum((-1, -2)).squeeze(-1)
        cum_nll_x = nll_x.cumsum(dim=1).diagonal(dim1=0, dim2=1)
        # cum_nll_x = total_nll_const_image(x).cumsum(dim=0)
        loss_beta = torch.relu(cum_nll_x - beta_incr) + torch.relu(beta_incr - cum_nll_x)

        # alternate
        if it % 2 == 0:
            loss = - dist.sum()
        else:
            loss = loss_beta.sum()
        # loss = - 1e2 * dist.sum() + 1e3 * loss_beta.sum()

        # maximize dist -> minimize -dist (unconstrained step)
        # loss = -dist
        loss.backward()
        torch.nn.utils.clip_grad_norm_([x], 1.0)
        outer_opt.step()
        with torch.no_grad():
            clamp_and_mask_(x)

        # # projection back to confidence set
        # if (it % proj_every) == 0:
        #     project_to_confidence_simple(x, step=step, steps=proj_steps, lr=1e-3)

        with torch.no_grad():
            # d_post = delta_total_nll(x)
            nll_post = nll(x.unsqueeze(1), counts_incr, intens_incr, angles).sum((-1, -2)).squeeze(-1)
            cum_nll_post = nll_post.cumsum(dim=1).diagonal(dim1=0, dim2=1)
            viol_post = torch.relu(cum_nll_post - beta_incr).cpu()
            history["nll"].append(cum_nll_post.cpu())
            history["dist"].append(dist.cpu())
            # history["delta"].append(float(d_post.cpu()))
            history["viol"].append(viol_post)

        if (it % 100) == 0 or it < 10:
            print(
                f"it={it:4d} step={print_step}, dist={history['dist'][-1][print_step]:.6f}  "
                f"NLL={history['nll'][-1][print_step].item():.3f}  beta={beta_incr[print_step].item():.3f}  viol={history['viol'][-1][print_step].item():.3e}"
            )

    # projection back to confidence set
    # project_to_confidence_simple(x, step=step, steps=proj_steps, lr=1e-2)

    maximizer = x.detach()
    return maximizer, history
print(gt_lr.shape, pred_incr.shape)
maximizer, history = compute_maximizer(initial_x=pred_incr.to(device) + 1. * torch.randn_like(pred_incr.to(device)), ref_x=pred_incr.to(device), print_step=50)
maximizer

In [ ]:
total_intensities

In [ ]:
plt.plot(total_intensities.squeeze().cumsum(dim=0)[1:].cpu().numpy(), history['dist'][-1].cpu().numpy())
plt.xscale('log')
# plt.yscale('log')

slope = (history['dist'][-1].cpu().numpy()[step] - history['dist'][0].cpu().numpy()[step]) / len(history['dist'])
slope

In [ ]:
# maximizer2 = compute_maximizer(initial_x=gt_lr, ref_x=maximizer)

In [ ]:
total_nll_gt

In [ ]:
step = 99


# --- final checks: total_nll + compare to beta ---
with torch.no_grad():
    total_nll_gt = nll(gt_lr, counts_incr, intens_incr, angles).sum((-1, -2, -3)).cumsum(dim=0)
    total_nll_max = nll(maximizer.unsqueeze(1), counts_incr, intens_incr, angles).sum((-1, -2, -3))
    total_nll_max = total_nll_max.cumsum(dim=1).diagonal(dim1=0, dim2=1)
    total_nll_pred = nll(pred_incr.to(device), counts_incr, intens_incr, angles).sum((-1, -2, -3)).cumsum(dim=0)
    # delta_max = total_nll_max - total_nll_gt[:step+1].sum()
    dist_max = (maximizer - pred_incr.to(device)).pow(2).mean(dim=(-1, -2, -3))

print("=== Final ===")
print("total_nll(gt_lr)      =", float(total_nll_gt[step].cpu()))
print("total_nll(maximizer)  =", float(total_nll_max[step].cpu()))
print("total_nll(pred)      =", float(total_nll_pred[step].cpu()))
# print("ΔNLL(maximizer)       =", float(delta_max[:step+1].sum().cpu()))
print("beta                  =", beta_incr[step].item())
print("feasible (total_nll_max <= beta) =", bool((total_nll_max[step] <= beta_incr[step]).item()))
print(f"delta_nll (maximizer - gt) =", float((total_nll_max[step] - total_nll_gt[step]).cpu()))
print("distance (MSE)        =", float(dist_max[step].cpu()))

In [ ]:
step = 50
# --- visualize result ---
maximizer_img = maximizer[step].squeeze().detach().cpu().numpy()
# maximizer2_img = maximizer2.squeeze().detach().cpu().numpy()
pred_img = pred_incr[step].squeeze().detach().cpu().numpy()
absdiff = np.abs(maximizer_img - pred_img)
error = np.abs(pred_img - gt_lr.squeeze().detach().cpu().numpy())

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].imshow(pred_img, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("pred_final")
axes[0].axis("off")

axes[1].imshow(maximizer_img, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("maximizer (in CS)")
axes[1].axis("off")

# axes[2].imshow(maximizer2_img, cmap="gray", vmin=0, vmax=1)
# axes[2].set_title("maximizer (in CS)")
# axes[2].axis("off")

axes[2].imshow(absdiff, cmap="magma")
axes[2].set_title("|maximizer - pred_final|")
axes[2].axis("off")
plt.colorbar(axes[2].images[0], ax=axes[2], fraction=0.046, pad=0.04)

axes[3].imshow(error, cmap="magma")
axes[3].set_title("|pred_final - gt_lr|")
axes[3].axis("off")
plt.colorbar(axes[3].images[0], ax=axes[3], fraction=0.046, pad=0.04)

# axes[3].plot(history["dist"], label="dist (MSE)")
# axes[3].plot(history["delta"], label="ΔNLL")
# axes[3].axhline(beta, color="k", linestyle="--", label="beta")
# axes[3].set_title("optimization trace")
# axes[3].legend()

plt.tight_layout()
plt.show()

In [ ]:
from cProfile import label
from tkinter import font
from uqct.evaluation.eval_dense import schedule_exponential, ObservationDataset, fbp_recon, psnr
from uqct.datasets.utils import get_dataset
from tqdm.auto import tqdm
import torch.nn.functional as F
import torch
import numpy as np
import matplotlib.pyplot as plt
from uqct.metrics import get_metrics
device = 'cuda'

_ , test_set = get_dataset('lung', True)

num_angles = 200
angles = torch.from_numpy(np.linspace(0, 180, num_angles, endpoint=False)).float().to(device)

schedule = schedule_exponential(
    1e9, 10, initial_intensity=1e6, device=device
)
total_intensities = schedule.clone()
schedule = schedule.reshape(-1, 1, 1, 1).expand(-1, 1, num_angles, 1) / num_angles / 256
    
obs_dataset = ObservationDataset(test_set, seeds=[0], intensities=schedule, angles=angles)
obs_dataloader = torch.utils.data.DataLoader(
    obs_dataset, batch_size=10, shuffle=False, num_workers=0
)


for indices, seed, images, data in tqdm(obs_dataloader):
    images = images.to(device)
    images_lr = F.interpolate(images, size=(128, 128), mode="area")
    data = data.to(device)

    data_cumsum = data.cumsum(dim=1)
    schedule_cumsum = schedule.unsqueeze(0).cumsum(dim=1)
    with torch.no_grad():
        pred = [fbp_recon(data_cumsum[:, i], schedule_cumsum[:, i], angles) for i in range(data.shape[1])]
    pred = torch.stack(pred, dim=-4)


    psnr_ = psnr(pred, images_lr.unsqueeze(1), circle_mask=True).squeeze(-1)
    psnr1_ = psnr(pred, images_lr.unsqueeze(1), data_range=1., circle_mask=True).squeeze(-1)

    for target, prediction in zip(images_lr, pred[:, -1]):
        print(get_metrics(target, prediction, data_range=1., circle_mask=True))

    fig, axes_ = plt.subplots(3, 3, figsize=(9, 9))
    for i, axes in enumerate(axes_):
        axes[0].imshow(images_lr[i, 0].cpu(), cmap='gray')
        axes[0].set_title("Ground Truth (LR)", fontsize=10)
        axes[0].axis('off')

        axes[1].imshow(pred[i, -1, 0].cpu(), cmap='gray')
        axes[1].set_title("FBP Recon (Final Step)", fontsize=10)
        axes[1].axis('off')

        axes[2].plot(total_intensities.cumsum(dim=0).cpu(), psnr_[i].squeeze().cpu().numpy(), label='psnr')
        axes[2].plot(total_intensities.cumsum(dim=0).cpu(), psnr1_[i].squeeze().cpu().numpy(), label='psnr (max_pixel=1)')
        axes[2].set_xscale('log')
        axes[2].set_ylabel('psnr')
        axes[2].set_xlabel('intensity')
        axes[2].set_title(f'Final PSNR: {psnr_[i, -1].squeeze().item():.2f} dB', fontsize=10)
        axes[2].legend()

    break
plt.show()
psnr_

In [ ]:
from uqct.evaluation.eval_dense import schedule_exponential, ObservationDataset, fbp_recon, psnr
from uqct.datasets.utils import get_dataset
_ , test_set = get_dataset('lung', True)
obs_dataset = ObservationDataset(test_set, seeds=[0], intensities=schedule, angles=angles)
obs_dataloader = torch.utils.data.DataLoader(
    obs_dataset, batch_size=20, shuffle=False, num_workers=0
)


for indices, seed, images, data in tqdm(obs_dataloader):
    images = images.to(device)
    images_lr = F.interpolate(images, size=(128, 128), mode="area")
    data = data.to(device)

    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    axes = axes.flatten()
    # plot images in grid
    for i in range(20):
        axes[i].imshow(images_lr[i, 0].cpu(), cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f"Image {i}")

    plt.show()
    break

In [ ]:
import os
path = '/mydata/chip/shared/data/lung/ground_truth_train'
folder = [
                filename
                for filename in os.listdir(path)
                if filename.endswith(".h5")
                or filename.endswith(".hdf5")
                or filename.endswith(".mat")
            ]

# sort
# folder = sorted(folder)

with open('lung_h5_files.txt', 'w') as f:
    for filename in folder:
        f.write(f"{filename}\n")

In [ ]:

path = '/mydata/chip/shared/data/lamino_tiff'

In [ ]:
from pathlib import Path
from typing import Literal

import torch
from torch.utils.data import Subset

from uqct.datasets.nii_tomogram_dataset import NiiDataset
from uqct.datasets.tiff_tomogram_dataset import TIFFDataset
from uqct.datasets.tomogram_dataset import TomogramDataset

DATA_DIR_CANDIDATES = [
    Path(x)
    for x in (
        "/mydata/chip/shared/data",
        "../data",
        "./data",
        "/cluster/scratch/mgaetzner/data",
    )
]
DATA_DIR = None
for x in DATA_DIR_CANDIDATES:
    if x.is_dir():
        DATA_DIR = x
if DATA_DIR is None:
    raise FileNotFoundError(
        f"Couldn't find data directory. Checked {DATA_DIR_CANDIDATES}"
    )

KWARGS_LAMINO = {
    "path": DATA_DIR / "lamino_tiff",
    "rescale": 128,
    "im_size": 256,
    "val_range": (0.0, 247.86),
    "rotation_angle": 30,
}

KWARGS_LUNG = {
    "path": DATA_DIR / "lung/ground_truth_train",
    "rescale": 128,
    "val_range": (0.0, 1.0),
    "rotation_angle": 30,
}

KWARGS_COMPOSITE = {
    "path": DATA_DIR / "composite/SampleG-FBI22-Stitch-0-1-2.txm.nii",
    "rescale": 128,
    "im_size": 256,
    "file_range": [20, 360],
    "val_range": [3e4, 4e4],
    "rotation_angle": 30,
}

DatasetName = Literal["composite", "lamino", "lung"]

def get_dataset(
    name: DatasetName, high_res: bool = False
) -> tuple[Subset[torch.Tensor], Subset[torch.Tensor]]:
    settings = {
        "composite": {"kwargs": KWARGS_COMPOSITE, "filetype": "nii"},
        "lamino": {"kwargs": KWARGS_LAMINO, "filetype": "tiff"},
        "lung": {"kwargs": KWARGS_LUNG, "filetype": "h5"},
    }

    # We need 256x256 to mitigate 'inverse crime problem'
    if high_res:
        for v in settings.values():
            v["kwargs"]["rescale"] = 256

    dataset_type = settings[name]["filetype"]
    kwargs = settings[name]["kwargs"]

    dataset_class = TomogramDataset if dataset_type == "h5" else TIFFDataset
    dataset_class = NiiDataset if dataset_type == "nii" else dataset_class

    if dataset_type == "tiff" and "im_size" not in kwargs:
        kwargs["im_size"] = 512

    if dataset_type == "nii" and "clip_range" not in kwargs:
        kwargs["im_size"] = 512

    dataset = dataset_class(**kwargs)
    torch.manual_seed(0)
    perm = torch.randperm(len(dataset))
    # Save permutation indices to a txt file
    perm_path =  f"{name}_perm.txt"
    np.savetxt(perm_path, perm.cpu().numpy(), fmt='%d')

get_dataset('lung')
get_dataset('lamino')

In [ ]:
from typing import Literal

import click
import torch

from uqct.ct import prepare_inputs_from_experiment, Experiment
from uqct.eval.run import run_evaluation
from uqct.eval.options import common_options

DatasetName = Literal["lung", "composite", "lamino"]


def run_fbp(
    dataset: DatasetName,
    sparse: bool,
    total_intensity: float,
    image_range: tuple[int, int],
    seed: int,
    n_angles: int,
    schedule_start: int,
    schedule_type: Literal["linear", "exp"],
    schedule_length: int,
    max_angle: int,
):
    def predictor_fn(
        experiment: Experiment, schedule: torch.Tensor | None
    ) -> torch.Tensor:
        # (N, T, H, W)
        preds, _, _ = prepare_inputs_from_experiment(experiment, schedule)
        return preds

    run_evaluation(
        dataset=dataset,
        sparse=sparse,
        total_intensity=total_intensity,
        image_range=image_range,
        seed=seed,
        model_name="fbp",
        predictor_fn=predictor_fn,
        n_angles=n_angles,
        schedule_start=schedule_start,
        schedule_type=schedule_type,
        schedule_length=schedule_length,
        max_angle=max_angle,
    )